# Assignment 2 — Regression and Classification: Error Analysis

**Holon Institute of Technology (HIT) — Faculty of Computer Science**

| | |
|---|---|
| **Course** | Introduction to Data Science |
| **Assignment** | 2 — Regression and Classification: Error Analysis |
| **Student name** | Sean Mazon |
| **ID number** | 212234660 |
| **Lecturer** | Dr. Uri Itai |
| **Teaching assistant** | Hanit Ohayon Hadad |
| **Submission deadline** | 10.09.2026 |

---

## Table of Contents

0. [Introduction](#0-introduction)
1. [Data Preparation](#1-data-preparation)
2. [Regression Error Analysis](#2-regression-error-analysis)
3. [Regression Models](#3-regression-models)
4. [Classification Error Analysis](#4-classification-error-analysis)
5. [Final Reflection](#5-final-reflection)

## 0. Introduction

This chapter sets out what the assignment asks for, what data the analysis is
built on, and which parts of the setup are prescribed by the assignment as
opposed to chosen here. It contains no code and no results; the analysis begins
in chapter 1.

### 0.1 Objective

The assignment defines its own purpose as developing an understanding of
regression and classification models **through systematic error analysis**, and
asks students to distinguish between failures that arise from data issues, from
model assumptions, and from how the problem was formulated.

That framing determines how this notebook is written. **Predictive accuracy is
not the goal.** A model that scored well but whose errors could not be
explained would fail the assignment, while a model that performs modestly and
whose failures are understood would satisfy it. Every metric reported here
exists to support a statement about *why* the model is wrong where it is wrong.

The assignment prescribes four things, and they are the fixed points of this
notebook:

1. The four analytical sections and their sub-tasks — residual analysis, error
   as a function of features, extreme errors, statistical properties of errors,
   the regression model comparison, the classification error analysis, and the
   final reflection.
2. Three families of regression model: linear, decision tree, and at least one
   ensemble.
3. That **all analyses be conducted using k-fold cross-validation**.
4. That the choice of *k* be justified in terms of dataset size, computational
   complexity, and the bias–variance trade-off.

Everything else — which dataset, which target variables, which features, which
classifiers — is not specified by the assignment. Those are decisions taken
here, and each one is identified as such and justified where it is made.

### 0.2 Dataset Overview

The assignment does not supply or name a dataset. This analysis continues with
the dataset used in Assignment 1: **1,500 of the highest-revenue games released
on Steam during 2024**, obtained from Kaggle and extracted on 2024-09-09. The
file holds 1,500 rows and 11 columns, mixing numeric, temporal and categorical
variables.

Using the same data is a deliberate choice. Assignment 1 established a set of
findings about its structure that directly constrain any model built on it, and
carrying those forward is more informative than rediscovering them:

| Finding from Assignment 1 | Consequence for this assignment |
|---|---|
| `reviewScore = 0` is a placeholder in 99 rows, `avgPlaytime = 0` in one | Must become missing values, then be imputed **inside** the cross-validation folds |
| `revenue` is extremely heavy-tailed (skew 22.9; 1.19 after a log transform) | Motivates the regression target chosen in 0.3 |
| `publishers` and `developers` hold 1,169 and 1,517 distinct companies | Cannot be one-hot encoded; must be aggregated into numeric features |
| `publisherClass` has a single `Hobbyist` row | Merged into `Indie`, leaving three ordered levels |
| Three missing cells, all in the company fields | Imputation strategy needed, though the volume is negligible |
| The rows were selected *because* revenue was high | The most important caveat in this notebook — see below |

**The survivorship caveat, stated at the outset.** These 1,500 games were
selected on revenue, which is the same quantity this notebook sets out to
predict and to classify. Every result that follows therefore describes the
behaviour of models *within an already-successful population*, never across
Steam as a whole. Statements such as "the model predicts commercial success"
must be read as "the model separates the strongest performers from the
moderately strong ones". This is not a limitation discovered at the end of the
analysis; it is a property of the sample and it is carried through every
chapter.

The full structural analysis of this dataset is in the Assignment 1 notebook,
`assignment1-eda/notebooks/steam_2024_eda.ipynb`.

### 0.3 Problem Definitions

**The assignment specifies neither a regression target nor a classification
target.** Both are defined here, and both are methodological choices rather than
requirements. Each is justified in full in chapter 1, where the supporting
evidence is computed; the definitions are stated here so the rest of the
notebook can refer to them.

#### Regression problem

> Predict **`log10(revenue)`** — the base-10 logarithm of a game's estimated
> revenue in US dollars.

The logarithm is used because Assignment 1 measured a skew of 22.9 on the raw
values against 1.19 after the transform. Residual analysis on the raw scale
would be dominated by a handful of titles and would make the questions in
section 2.1 — about centring, systematic pattern and heteroscedasticity —
effectively unanswerable. Metrics are also reported back on the original dollar
scale so that the error magnitudes remain interpretable.

#### Classification problem

> Predict whether a game is a **commercial standout**, defined as
> `revenue` at or above the 75th percentile of the dataset.

This yields roughly 375 positive and 1,125 negative cases. A threshold at the
median would produce a balanced problem, but a 25/75 split is the more
informative choice for what section 3 of the assignment asks: it gives the
precision–recall trade-off something to trade, and it makes the threshold sweep
and the Matthews correlation coefficient meaningful rather than near-symmetric.

The two error types also carry genuinely different costs in the underlying
decision problem, which section 4.2 develops. A false positive corresponds to
backing a title that does not deliver; a false negative corresponds to passing
on one that would have. The threshold is a fixed definitional choice computed
once on the full dataset, not a quantity learned from the data.

#### A note on the feature set

The features are **not fixed at this point**. Assignment 1 found that `revenue`,
`copiesSold` and `price` are tightly linked, which raises the possibility that
including `copiesSold` as a predictor would make both problems trivial and leave
no error structure to analyse. Section 1 tests that empirically and decides the
feature set on the evidence, rather than assuming the answer here.

### 0.4 Methodology

Each analytical section follows the same four-part pattern:

- **Method** — what is being done and why.
- **Results** — the metrics, tables and figures.
- **Interpretation** — what the results mean, including where they contradict
  what was expected.
- **Conclusion** — the single insight the section establishes.

Two rules govern the writing. First, **interpretations are written after the
results are read**, never drafted in advance; where an outcome is surprising it
is reported as surprising rather than smoothed over. Second, **evidence and
hypothesis are kept distinct**: where the data shows a pattern but cannot
establish its cause, the proposed cause is labelled as a hypothesis and the
information that would test it is named.

Chapter 3 of the assignment requires three families of regression model. This
notebook uses Linear Regression, a Decision Tree Regressor, and a **Random
Forest Regressor** as the ensemble. All three are trained on an identical
feature set under an identical resampling protocol, so that differences in
performance are attributable to the models rather than to their inputs.

For the classification analysis the assignment names no model. Two are used
here — **Logistic Regression** and a **Random Forest Classifier** — because the
discussion section calls for a comparison, and a comparison needs more than one
subject. This is a choice, not a requirement.

### 0.5 Cross-Validation Strategy

The assignment requires that **all** analyses use k-fold cross-validation. That
requirement is stated in its objective rather than in the modelling section, so
it governs the error analysis as much as the model comparison: the residuals
examined in chapter 2 and the confusion matrix in chapter 4 must not be computed
from predictions the model made on data it was trained on.

The protocol used throughout is therefore:

- **Out-of-fold prediction.** Every observation receives a prediction from a
  model that never saw it during training. Pooling these across folds yields one
  prediction per row, which is what the residual analysis, the extreme-error
  analysis and the confusion matrix are built from.
- **k = 10**, with shuffling and a fixed random seed. The justification in terms
  of dataset size, computational cost and the bias–variance trade-off is given
  in section 1, where it can be supported with the actual dimensions of the data
  rather than asserted here.
- **Stratified folds for classification**, so that the 25/75 class balance is
  preserved in every fold and the per-fold metrics remain comparable.
- **Pipelines for all preprocessing.** Imputation, scaling and encoding are
  fitted on the training portion of each fold only. Fitting them on the full
  dataset first — the most common source of leakage in cross-validated work —
  would let information from the held-out fold influence the transformation
  applied to it.

Section 1 sets this protocol up and states explicitly what each element
prevents.

## 1. Data Preparation

*Not yet written.* Loading, cleaning carried over from Assignment 1, the empirical leakage test that determines the feature set, the justification of *k*, and the pipeline design.

## 2. Regression Error Analysis

*Not yet written.* Residual analysis, error as a function of features, the top 5% of absolute errors, and the statistical properties of the error distribution.

## 3. Regression Models

*Not yet written.* Linear Regression, Decision Tree and Random Forest compared on a common feature set, followed by the critical discussion and the choice of a preferred model.

## 4. Classification Error Analysis

*Not yet written.* Confusion matrix, false positives and negatives, probability-based analysis, error as a function of features, threshold sensitivity, and ROC/AUC with the Matthews correlation coefficient.

## 5. Final Reflection

*Not yet written.* The four closing questions, answered explicitly.